# Giani anchor render — LatentSync 1.6

Private Kaggle render worker: fetch a canonical anchor clip and voice from short-lived Cloudflare R2 signed URLs, create a duration-matched ping-pong bed, run ByteDance LatentSync 1.6, generate SRT captions with faster-whisper, validate outputs, and optionally upload them through signed PUT URLs.

**Hardware truth:** [ByteDance's official README](https://github.com/bytedance/LatentSync) states that LatentSync 1.6 needs at least **18 GB VRAM**. A T4 has 16 GB; two T4s do not pool memory for the official single-GPU inference command. Use a Kaggle accelerator with at least 18 GB on one device when available. This notebook never attempts 1.6 below that limit. It does not silently downgrade because a safe LatentSync 1.5 checkpoint/config path is not implemented here; use the documented MuseTalk fallback instead.

Keep this notebook private. Signed URLs are secrets even though they expire.

In [ ]:
# Parameter cell — safe values only. Secret *names*, never secret values, live here.
PARAMS = {
    "latentsync_repo": "https://github.com/bytedance/LatentSync.git",
    "latentsync_ref": "a229c3948406bc2cf6eaf4873e662e70c6a04746",
    "video_get_url_secret": "R2_ANCHOR_VIDEO_GET_URL",
    "audio_get_url_secret": "R2_VOICE_AUDIO_GET_URL",
    "video_put_url_secret": "R2_SYNCED_VIDEO_PUT_URL",
    "captions_put_url_secret": "R2_CAPTIONS_PUT_URL",
    "manifest_put_url_secret": "R2_RENDER_MANIFEST_PUT_URL",
    "inference_steps": 20,
    "guidance_scale": 1.5,
    "seed": 1247,
    "enable_deepcache": True,
    "whisper_model": "small.en",
    "whisper_language": "en",
    "max_caption_words": 6,
    "max_caption_seconds": 2.8,
    "minimum_vram_gib": 18.0,
    "request_timeout_seconds": 120,
}


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import pathlib
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from urllib.parse import urlparse

from kaggle_secrets import UserSecretsClient

WORK = pathlib.Path("/kaggle/working/giani_render")
INPUT = WORK / "input"
OUTPUT = pathlib.Path("/kaggle/working/giani_output")
REPO = pathlib.Path("/kaggle/working/LatentSync")
for directory in (WORK, INPUT, OUTPUT):
    directory.mkdir(parents=True, exist_ok=True)

def run(command: list[str], *, cwd: pathlib.Path | None = None) -> subprocess.CompletedProcess[str]:
    print("+", " ".join(command))
    return subprocess.run(command, cwd=cwd, text=True, check=True)

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No NVIDIA GPU is attached. In Kaggle: Notebook options > Accelerator > GPU.")

gpu_query = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip().splitlines()
gpus = []
for row in gpu_query:
    name, memory_mib = row.rsplit(",", 1)
    gpus.append({"name": name.strip(), "memory_mib": int(memory_mib.strip())})
best_vram_gib = max(gpu["memory_mib"] for gpu in gpus) / 1024
print("Detected GPUs:", [{"name": g["name"], "vram_gib": round(g["memory_mib"] / 1024, 1)} for g in gpus])
if best_vram_gib < float(PARAMS["minimum_vram_gib"]):
    raise RuntimeError(
        f"Best single GPU has {best_vram_gib:.1f} GiB. ByteDance documents at least "
        f"{PARAMS['minimum_vram_gib']:.0f} GiB for LatentSync 1.6. A T4 x2 session still has only "
        "16 GiB per process. This notebook does not implement the separate LatentSync 1.5 "
        "checkpoint/config path, so it will not silently downgrade. Select a >=18 GiB GPU, or "
        "run MuseTalk as the supported 16 GiB fallback described in infra/README.md."
    )

secret_client = UserSecretsClient()
def read_secret(name: str, *, required: bool) -> str | None:
    try:
        value = secret_client.get_secret(name)
    except Exception as exc:
        if required:
            raise RuntimeError(f"Missing Kaggle Secret {name!r}. Add it in Add-ons > Secrets and attach it to this notebook.") from exc
        return None
    value = (value or "").strip()
    if required and not value:
        raise RuntimeError(f"Kaggle Secret {name!r} is empty.")
    if value and urlparse(value).scheme != "https":
        raise RuntimeError(f"Kaggle Secret {name!r} must contain an HTTPS signed URL.")
    return value or None

SIGNED_URLS = {
    "video_get": read_secret(PARAMS["video_get_url_secret"], required=True),
    "audio_get": read_secret(PARAMS["audio_get_url_secret"], required=True),
    "video_put": read_secret(PARAMS["video_put_url_secret"], required=False),
    "captions_put": read_secret(PARAMS["captions_put_url_secret"], required=False),
    "manifest_put": read_secret(PARAMS["manifest_put_url_secret"], required=False),
}
print("Required signed GET URLs loaded; optional PUT URLs configured:", {key: bool(value) for key, value in SIGNED_URLS.items() if key.endswith("put")})


## Install the official implementation

The clone is pinned to a reviewed ByteDance commit. The checkpoint downloads and inference arguments below mirror ByteDance's official `setup_env.sh` and `inference.sh`; Kaggle's managed Python kernel replaces the official Conda environment.

In [ ]:
if not (REPO / ".git").exists():
    run(["git", "clone", PARAMS["latentsync_repo"], str(REPO)])
run(["git", "fetch", "--depth", "1", "origin", PARAMS["latentsync_ref"]], cwd=REPO)
run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO)

run(["apt-get", "update", "-qq"])
system_packages = ["libgl1"]
if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:
    system_packages.append("ffmpeg")
run(["apt-get", "install", "-y", "-qq", *system_packages])

# ByteDance official environment installs requirements.txt. Extra packages are
# the caption/R2 layer owned by this notebook.
run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-r", "requirements.txt"], cwd=REPO)
run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "faster-whisper==1.2.0", "requests==2.32.4"])

run([
    "huggingface-cli", "download", "ByteDance/LatentSync-1.6", "whisper/tiny.pt",
    "--local-dir", "checkpoints",
], cwd=REPO)
run([
    "huggingface-cli", "download", "ByteDance/LatentSync-1.6", "latentsync_unet.pt",
    "--local-dir", "checkpoints",
], cwd=REPO)

for required in (REPO / "checkpoints/whisper/tiny.pt", REPO / "checkpoints/latentsync_unet.pt"):
    if not required.is_file() or required.stat().st_size < 1_000_000:
        raise RuntimeError(f"Checkpoint missing or incomplete: {required}")
print("Official LatentSync source and inference checkpoints are ready.")


In [ ]:
import requests

def stream_download(url: str, destination: pathlib.Path) -> None:
    temporary = destination.with_suffix(destination.suffix + ".partial")
    with requests.get(url, stream=True, timeout=PARAMS["request_timeout_seconds"], allow_redirects=True) as response:
        response.raise_for_status()
        with temporary.open("wb") as handle:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    handle.write(chunk)
    if temporary.stat().st_size == 0:
        raise RuntimeError(f"Signed download produced an empty file: {destination.name}")
    temporary.replace(destination)

anchor_source = INPUT / "anchor_source.mp4"
voice_source = INPUT / "voice_source"
stream_download(SIGNED_URLS["video_get"], anchor_source)
stream_download(SIGNED_URLS["audio_get"], voice_source)
print("Downloaded signed inputs without logging their URLs:", anchor_source.stat().st_size, voice_source.stat().st_size, "bytes")


In [ ]:
def ffprobe(path: pathlib.Path) -> dict:
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-show_streams", "-show_format", "-of", "json", str(path)],
        check=True,
        capture_output=True,
        text=True,
    )
    return json.loads(result.stdout)

def media_duration(probe: dict) -> float:
    duration = float(probe.get("format", {}).get("duration") or 0)
    if not (duration > 0):
        raise RuntimeError("ffprobe did not report a positive media duration.")
    return duration

anchor_probe = ffprobe(anchor_source)
voice_probe = ffprobe(voice_source)
if not any(stream.get("codec_type") == "video" for stream in anchor_probe["streams"]):
    raise RuntimeError("Anchor signed URL did not return a video stream.")
if not any(stream.get("codec_type") == "audio" for stream in voice_probe["streams"]):
    raise RuntimeError("Voice signed URL did not return an audio stream.")

voice_wav = WORK / "voice_16k_mono.wav"
run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-i", str(voice_source), "-vn", "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", str(voice_wav)])
voice_duration = media_duration(ffprobe(voice_wav))

pingpong = WORK / "anchor_pingpong.mp4"
run([
    "ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-i", str(anchor_source),
    "-filter_complex", "[0:v]fps=25,split=2[forward][reverse_in];[reverse_in]reverse[reverse];[forward][reverse]concat=n=2:v=1:a=0,format=yuv420p[out]",
    "-map", "[out]", "-an", "-c:v", "libx264", "-crf", "18", "-preset", "medium", str(pingpong),
])

anchor_bed = WORK / "anchor_bed.mp4"
run([
    "ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-stream_loop", "-1", "-i", str(pingpong),
    "-t", f"{voice_duration:.3f}", "-an", "-r", "25", "-c:v", "libx264", "-crf", "18", "-preset", "medium", "-pix_fmt", "yuv420p", str(anchor_bed),
])
bed_duration = media_duration(ffprobe(anchor_bed))
if abs(bed_duration - voice_duration) > 0.25:
    raise RuntimeError(f"Duration-matched bed differs from voice by {abs(bed_duration - voice_duration):.3f}s")
print(f"Validated duration-matched anchor bed: {bed_duration:.3f}s for {voice_duration:.3f}s voice")


In [ ]:
# ByteDance's official inference.sh command, parameterized only for this job.
raw_synced = WORK / "anchor_synced_raw.mp4"
inference_command = [
    sys.executable, "-m", "scripts.inference",
    "--unet_config_path", "configs/unet/stage2_512.yaml",
    "--inference_ckpt_path", "checkpoints/latentsync_unet.pt",
    "--inference_steps", str(PARAMS["inference_steps"]),
    "--guidance_scale", str(PARAMS["guidance_scale"]),
    "--video_path", str(anchor_bed),
    "--audio_path", str(voice_wav),
    "--video_out_path", str(raw_synced),
    "--seed", str(PARAMS["seed"]),
]
if PARAMS["enable_deepcache"]:
    inference_command.append("--enable_deepcache")
run(inference_command, cwd=REPO)
if not raw_synced.is_file() or raw_synced.stat().st_size == 0:
    raise RuntimeError("LatentSync completed without a non-empty output video.")


In [ ]:
synced_video = OUTPUT / "anchor_synced.mp4"
run([
    "ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-i", str(raw_synced), "-i", str(voice_wav),
    "-map", "0:v:0", "-map", "1:a:0", "-c:v", "copy", "-c:a", "aac", "-b:a", "192k", "-shortest", "-movflags", "+faststart", str(synced_video),
])
synced_probe = ffprobe(synced_video)
if not any(stream.get("codec_type") == "video" for stream in synced_probe["streams"]):
    raise RuntimeError("Final synced output has no video stream.")
if not any(stream.get("codec_type") == "audio" for stream in synced_probe["streams"]):
    raise RuntimeError("Final synced output has no audio stream.")
synced_duration = media_duration(synced_probe)
if abs(synced_duration - voice_duration) > 0.5:
    raise RuntimeError(f"Synced output duration mismatch: video={synced_duration:.3f}s voice={voice_duration:.3f}s")
print(f"Validated LatentSync output: {synced_duration:.3f}s")


In [ ]:
from faster_whisper import WhisperModel

whisper = WhisperModel(PARAMS["whisper_model"], device="cuda", compute_type="float16")
segments_iter, transcription_info = whisper.transcribe(
    str(voice_wav),
    language=PARAMS["whisper_language"],
    beam_size=5,
    vad_filter=True,
    word_timestamps=True,
)
segments = list(segments_iter)
words = [word for segment in segments for word in (segment.words or []) if word.word.strip()]
if not words:
    raise RuntimeError("faster-whisper returned no timestamped words; captions cannot be generated.")

def srt_time(seconds: float) -> str:
    milliseconds = max(0, round(seconds * 1000))
    hours, remainder = divmod(milliseconds, 3_600_000)
    minutes, remainder = divmod(remainder, 60_000)
    secs, millis = divmod(remainder, 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"

caption_groups = []
current = []
for word in words:
    candidate_duration = float(word.end) - (float(current[0].start) if current else float(word.start))
    if current and (len(current) >= int(PARAMS["max_caption_words"]) or candidate_duration > float(PARAMS["max_caption_seconds"])):
        caption_groups.append(current)
        current = []
    current.append(word)
if current:
    caption_groups.append(current)

captions_path = OUTPUT / "subs.srt"
with captions_path.open("w", encoding="utf-8", newline="\n") as handle:
    for index, group in enumerate(caption_groups, start=1):
        text = "".join(word.word for word in group).strip().replace("-->", "→")
        handle.write(f"{index}\n{srt_time(float(group[0].start))} --> {srt_time(float(group[-1].end))}\n{text}\n\n")
if captions_path.stat().st_size == 0:
    raise RuntimeError("SRT generation produced an empty file.")
print(f"Generated {len(caption_groups)} caption cues at {captions_path}")


In [ ]:
def sha256_file(path: pathlib.Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def url_fingerprint(url: str) -> str:
    return hashlib.sha256(url.encode("utf-8")).hexdigest()

manifest = {
    "schema_version": 1,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "renderer": {
        "repository": PARAMS["latentsync_repo"],
        "commit": PARAMS["latentsync_ref"],
        "model": "ByteDance/LatentSync-1.6",
        "config": "configs/unet/stage2_512.yaml",
        "inference_steps": PARAMS["inference_steps"],
        "guidance_scale": PARAMS["guidance_scale"],
        "seed": PARAMS["seed"],
        "deepcache": PARAMS["enable_deepcache"],
    },
    "hardware": {
        "gpus": [{"name": gpu["name"], "memory_mib": gpu["memory_mib"]} for gpu in gpus],
        "official_minimum_vram_gib": PARAMS["minimum_vram_gib"],
    },
    "inputs": {
        "anchor": {"sha256": sha256_file(anchor_source), "bytes": anchor_source.stat().st_size, "signed_url_fingerprint": url_fingerprint(SIGNED_URLS["video_get"])},
        "voice": {"sha256": sha256_file(voice_source), "bytes": voice_source.stat().st_size, "duration_seconds": voice_duration, "signed_url_fingerprint": url_fingerprint(SIGNED_URLS["audio_get"])},
    },
    "outputs": {
        "video": {"file": synced_video.name, "sha256": sha256_file(synced_video), "bytes": synced_video.stat().st_size, "duration_seconds": synced_duration},
        "captions": {"file": captions_path.name, "sha256": sha256_file(captions_path), "bytes": captions_path.stat().st_size, "cues": len(caption_groups)},
    },
    "uploads": {},
    "warnings": [],
}

def signed_put(url: str | None, path: pathlib.Path, content_type: str) -> dict:
    if not url:
        return {"status": "not_configured"}
    with path.open("rb") as handle:
        response = requests.put(
            url,
            data=handle,
            headers={"Content-Type": content_type},
            timeout=PARAMS["request_timeout_seconds"],
        )
    response.raise_for_status()
    return {"status": "uploaded", "http_status": response.status_code, "signed_url_fingerprint": url_fingerprint(url)}

manifest["uploads"]["video"] = signed_put(SIGNED_URLS["video_put"], synced_video, "video/mp4")
manifest["uploads"]["captions"] = signed_put(SIGNED_URLS["captions_put"], captions_path, "application/x-subrip; charset=utf-8")
manifest_path = OUTPUT / "render_manifest.json"
manifest["uploads"]["manifest"] = {"status": "pending"} if SIGNED_URLS["manifest_put"] else {"status": "not_configured"}
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
if SIGNED_URLS["manifest_put"]:
    manifest_upload = signed_put(SIGNED_URLS["manifest_put"], manifest_path, "application/json")
    manifest["uploads"]["manifest"] = manifest_upload
    manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    signed_put(SIGNED_URLS["manifest_put"], manifest_path, "application/json")

print("Render complete. Kaggle output artifacts:")
for path in (synced_video, captions_path, manifest_path):
    print(f"- {path} ({path.stat().st_size} bytes)")
print("Upload states:", {name: state["status"] for name, state in manifest["uploads"].items()})


## Operator handoff

1. Inspect `anchor_synced.mp4`, especially jaw/teeth transitions.
2. Inspect `subs.srt` for names, numbers, and acronyms.
3. Download the Kaggle output or use the optional signed PUT results.
4. Feed the synced video, original voice, captions, and a reviewed cue JSON into the local assembly script.

A successful render is not approval to publish; the human editorial and compliance review still applies.